# CNN Baseline — emg2qwerty TDSConvCTC\n\nBaseline training run using the original `log_spectrogram` pipeline:\n- **16 channels/hand**, **2000 Hz**, no filtering or downsampling\n- Model: TDSConvCTC (5.3M params)\n- Pipeline: `ToTensor → BandRotation → TemporalJitter → LogSpectrogram → SpecAugment`\n\nAll output logs and checkpoints are saved to `logs/YYYY-MM-DD/HH-MM-SS/`.

## 0. Setup\n\nMake sure you are running with the project venv active and installed:\n```bash\nsource /home/benforbes/emg2qwerty/venv/bin/activate\npip install -e .\n```\nThen launch Jupyter from the repo root:\n```bash\ncd /home/benforbes/C247_mumbikaijonathanben\njupyter notebook\n```

In [ ]:
import os, sys\n\n# Ensure we're running from the repo root\nREPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), \"..\", \"..\"))\nif REPO_ROOT not in sys.path:\n    sys.path.insert(0, REPO_ROOT)\nos.chdir(REPO_ROOT)\nprint(f\"Working directory: {os.getcwd()}\")

## 1. Patch emg2qwerty with Playground_Ben versions

In [ ]:
import shutil\nfrom pathlib import Path\n\nPLAYGROUND = Path(REPO_ROOT) / \"Playground_Ben\"\n\n# Overwrite the main emg2qwerty module with Playground_Ben versions\nshutil.copy(PLAYGROUND / \"emg2qwerty/transforms.py\", \"emg2qwerty/transforms.py\")\nshutil.copy(PLAYGROUND / \"emg2qwerty/lightning.py\",  \"emg2qwerty/lightning.py\")\nprint(\"Patched: emg2qwerty/transforms.py\")\nprint(\"Patched: emg2qwerty/lightning.py\")

## 2. Train — CNN Baseline (16 ch, 2000 Hz, log_spectrogram)

In [ ]:
import subprocess, sys\n\ncmd = [\n    sys.executable, \"-m\", \"emg2qwerty.train\",\n    \"user=single_user\",\n    # transforms defaults to log_spectrogram (baseline — no overrides needed)\n]\n\nprint(\"Running:\", \" \".join(cmd))\nresult = subprocess.run(cmd, cwd=REPO_ROOT)\nprint(f\"\\nExit code: {result.returncode}\")

## 3. Find the latest run and load results

In [ ]:
import glob\nfrom pathlib import Path\nfrom tensorboard.backend.event_processing.event_accumulator import EventAccumulator, SCALARS\n\n# Auto-detect the most recent log dir\nlogs = sorted(Path(REPO_ROOT, \"logs\").glob(\"*/*\"), reverse=True)\nlog_dir = next(\n    d for d in logs\n    if (d / \"lightning_logs\" / \"version_0\").exists()\n)\nprint(f\"Latest run: {log_dir}\")\n\n# Load TensorBoard scalars\ntb_dir = log_dir / \"lightning_logs\" / \"version_0\"\nevent_files = sorted(glob.glob(str(tb_dir / \"events.out.tfevents.*\")))\n\nscalars = {}\nfor ef in event_files:\n    ea = EventAccumulator(ef, size_guidance={SCALARS: 0})\n    ea.Reload()\n    for tag in ea.Tags().get(\"scalars\", []):\n        scalars.setdefault(tag, [])\n        scalars[tag] += ea.Scalars(tag)\n\nprint(f\"Tracked metrics: {list(scalars.keys())}\")

## 4. Plot training curves

In [ ]:
import numpy as np\nimport matplotlib.pyplot as plt\n\nUCLA = {\"blue\": \"#2774AE\", \"gold\": \"#FFD100\", \"dark_blue\": \"#003B5C\", \"dark_gold\": \"#FFB300\"}\n\ndef epoch_series(scalars, tag):\n    step_to_epoch = {e.step: int(e.value) for e in scalars.get(\"epoch\", [])}\n    pairs = [(step_to_epoch[e.step], e.value)\n             for e in scalars.get(tag, []) if e.step in step_to_epoch]\n    if not pairs:\n        return np.array([]), np.array([])\n    pairs.sort()\n    eps, vals = zip(*pairs)\n    return np.array(eps), np.array(vals)\n\nval_ep,   val_cer   = epoch_series(scalars, \"val/CER\")\nloss_ep,  val_loss  = epoch_series(scalars, \"val/loss\")\ntrain_ep, train_loss = epoch_series(scalars, \"train/loss\")\n\nfig, axes = plt.subplots(1, 2, figsize=(13, 4))\nfig.suptitle(\"CNN Baseline — 16 ch/hand, 2000 Hz\", fontsize=13)\n\n# CER\naxes[0].plot(val_ep, val_cer, color=UCLA[\"blue\"], lw=2, label=\"Val CER\")\naxes[0].set_xlabel(\"Epoch\")\naxes[0].set_ylabel(\"Character Error Rate (%)\")\naxes[0].set_title(\"Validation CER\")\naxes[0].set_ylim(bottom=0)\naxes[0].legend()\naxes[0].grid(True, alpha=0.25, linestyle=\"--\")\n\n# Loss\naxes[1].plot(train_ep, train_loss, color=UCLA[\"dark_blue\"], lw=1.5, alpha=0.8, label=\"Train loss\")\naxes[1].plot(loss_ep,  val_loss,   color=UCLA[\"gold\"],      lw=2,   label=\"Val loss\")\naxes[1].set_xlabel(\"Epoch\")\naxes[1].set_ylabel(\"CTC Loss\")\naxes[1].set_title(\"Train vs. Val Loss\")\naxes[1].set_ylim(bottom=0)\naxes[1].legend()\naxes[1].grid(True, alpha=0.25, linestyle=\"--\")\n\nplt.tight_layout()\nplt.show()

## 5. Final metrics summary

In [ ]:
def last_value(scalars, tag):\n    events = scalars.get(tag, [])\n    return events[-1].value if events else float(\"nan\")\n\nwall_times = [e.wall_time for evts in scalars.values() for e in evts]\ntraining_hrs = (max(wall_times) - min(wall_times)) / 3600 if wall_times else 0\n\nprint(\"=\" * 40)\nprint(\"  CNN Baseline Results\")\nprint(\"=\" * 40)\nprint(f\"  Val  CER  : {last_value(scalars, 'val/CER'):.2f}%\")\nprint(f\"  Test CER  : {last_value(scalars, 'test/CER'):.2f}%\")\nprint(f\"  Val  Loss : {last_value(scalars, 'val/loss'):.4f}\")\nprint(f\"  Train Loss: {last_value(scalars, 'train/loss'):.4f}\")\nprint(f\"  Training  : {training_hrs:.1f} hrs\")\nprint(\"=\" * 40)

## 6. Log to team results CSV

In [ ]:
sys.path.insert(0, str(PLAYGROUND / \"scripts\"))\nimport importlib, log_temporal_results\nimportlib.reload(log_temporal_results)\n\n# Run the logger — picks up the latest completed run automatically\n%run {PLAYGROUND}/scripts/log_temporal_results.py\nprint(\"Logged to results/results_summary_CNN.csv\")